In [1]:
from matplotlib import pyplot as plt
from matplotlib import colors
import numpy as np
from matplotlib.patches import Path, PathPatch
import pandas as pd
from shapely.geometry import Point, shape, Polygon
from shapely.ops import unary_union, cascaded_union
from geopandas.tools import sjoin
import geopandas as gpd
from netCDF4 import Dataset
from cartopy import crs as ccrs
from cartopy.io.shapereader import Reader
from sklearn.metrics import mean_squared_error
import scipy.stats as st
from sklearn.linear_model import LinearRegression
import cartopy.feature as cfeature
import csv  
from matplotlib.colors import ListedColormap
from cartopy.feature import LAKES

C:\Users\x12la\AppData\Local\Temp\ipykernel_11796\2511498831.py:8: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  from geopandas.tools import sjoin


<h1> Load and modify EPA files to merge with telemetrics data 

In [2]:
# print statements without sci notation for QA
pd.set_option("display.float_format", "{:,.6f}".format)

# Load EPA files 
oni_file = pd.read_csv('ONI_2022v1_projected_from_full_annual_20240226_monthly_18jun2024_nf_v5.csv',skiprows=14)
hotelling_file = pd.read_csv('HOTELING_2022v1_monthly_20240305_18jun2024_nf_v5.csv',skiprows=19)

# Col headers from EPA Files
cols = [
    "country_cd", "region_cd", "tribal_code", "census_tract_cd", "shape_id", "scc",
    "act_parm_type_cd", "act_parm_uofmsr", "activity_type", "ann_parm_value",
    "calc_year", "date_updated", "data_set_id",
    "jan_value", "feb_value", "mar_value", "apr_value", "may_value", "jun_value",
    "jul_value", "aug_value", "sep_value", "oct_value", "nov_value", "dec_value",
    "comment"
]
oni_file.columns = cols
hotelling_file.columns = cols

# Format data to make merging a bit easier later on
oni_file["region_cd"]    = oni_file["region_cd"].astype(str).str.zfill(5)
hotelling_file["region_cd"] = hotelling_file["region_cd"].astype(str).str.zfill(5)
scc_oni = oni_file["scc"].astype(str).str.zfill(10)
oni_file["veh_code"]  = scc_oni.str[4:6]
scc_hot = hotelling_file["scc"].astype(str).str.zfill(10)
hotelling_file["veh_code"]  = scc_hot.str[4:6]

# EPA calculates total idling hours, as well as hours per month.
# Here, I calc the ratio of monthly idling data - so that when we scale
# the data, it reflects the same temporal distribution as the original
# EPA input files.
month_cols = [
    "jan_value","feb_value","mar_value","apr_value","may_value","jun_value",
    "jul_value","aug_value","sep_value","oct_value","nov_value","dec_value"
]
ratio_cols = [c.replace("_value","_ratio") for c in month_cols]

for df in (oni_file, hotelling_file):
    df[month_cols] = df[month_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df["ann_parm_value"] = pd.to_numeric(df["ann_parm_value"], errors="coerce").fillna(0.0)

for df in (oni_file, hotelling_file):
    denom = df["ann_parm_value"].replace(0, np.nan)
    for m in month_cols:
        df[m.replace("_value", "_ratio")] = (df[m] / denom).fillna(0.0)

In [3]:
# Including print statements for sanity checks and hand calculations to check work
oni_file.loc[(oni_file['veh_code']=='52') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
11858,US,17031,NaN,NaN,NaN,2201520192,NaN,NaN,IDLING,"5,263,398.530000",...,0.087989,0.084330,0.087921,0.087870,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220
11868,US,17031,NaN,NaN,NaN,2202520192,NaN,NaN,IDLING,"8,632,081.771000",...,0.087989,0.084330,0.087921,0.087870,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220
11874,US,17031,NaN,NaN,NaN,2203520192,NaN,NaN,IDLING,"133,976.576900",...,0.087989,0.084330,0.087921,0.087870,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220


In [4]:
oni_file.loc[(oni_file['veh_code']=='53') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
11859,US,17031,NaN,NaN,NaN,2201530192,NaN,NaN,IDLING,"275,409.436300",...,0.087992,0.084323,0.087919,0.087872,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217
11869,US,17031,NaN,NaN,NaN,2202530192,NaN,NaN,IDLING,"687,557.819200",...,0.087992,0.084323,0.087919,0.087872,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217
11875,US,17031,NaN,NaN,NaN,2203530192,NaN,NaN,IDLING,"5,716.935015",...,0.087992,0.084323,0.087919,0.087872,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217


In [5]:
oni_file.loc[(oni_file['veh_code']=='61') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
11860,US,17031,NaN,NaN,NaN,2201610192,NaN,NaN,IDLING,"4,243.243977",...,0.088021,0.085634,0.087519,0.088470,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085
11870,US,17031,NaN,NaN,NaN,2202610192,NaN,NaN,IDLING,"4,871,835.012000",...,0.088021,0.085634,0.087519,0.088470,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085
11876,US,17031,NaN,NaN,NaN,2203610192,NaN,NaN,IDLING,"49,991.035370",...,0.088021,0.085634,0.087519,0.088470,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085


In [6]:
hotelling_file.loc[hotelling_file['region_cd']=='17031'].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
469,US,17031,NaN,NaN,NaN,2202620153,NaN,NaN,HOTELLING,"3,500,373.360000",...,0.087127,0.086217,0.086941,0.087149,0.083849,0.088669,0.082181,0.090221,0.082809,0.073965
470,US,17031,NaN,NaN,NaN,2202620191,NaN,NaN,HOTELLING,"380,306.640000",...,0.087127,0.086217,0.086941,0.087149,0.083849,0.088669,0.082181,0.090221,0.082809,0.073965


In [7]:
# Make copies so not to overwrite original files
# Here, I'm defining MDVs by vehicle class 52/53 
# (single unit trucks within the moves model)
mdv_oni = oni_file[oni_file["veh_code"].isin(["52","53"])].copy()
mdv_oni["ann_parm_value"] = pd.to_numeric(mdv_oni["ann_parm_value"], errors="coerce").fillna(0.0)

# HDVs are defined here vehicle code 61/62 (combination trucks)
hdv_oni = oni_file[oni_file["veh_code"].isin(["61","62"])].copy()
hdv_hot = hotelling_file[hotelling_file["veh_code"].isin(["61","62"])].copy()  

In [8]:
mdv_oni.head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
4,US,01001,NaN,NaN,NaN,2201520192,NaN,NaN,IDLING,"104,112.320800",...,0.069847,0.083470,0.091061,0.089665,0.092922,0.089711,0.088081,0.090094,0.089117,0.085722
5,US,01001,NaN,NaN,NaN,2201530192,NaN,NaN,IDLING,"23,933.198480",...,0.069847,0.083470,0.091061,0.089665,0.092922,0.089711,0.088081,0.090094,0.089117,0.085722
13,US,01001,NaN,NaN,NaN,2202520192,NaN,NaN,IDLING,"264,909.493100",...,0.069847,0.083470,0.091061,0.089665,0.092922,0.089711,0.088081,0.090094,0.089117,0.085722
14,US,01001,NaN,NaN,NaN,2202530192,NaN,NaN,IDLING,"55,790.871330",...,0.069847,0.083470,0.091061,0.089665,0.092922,0.089711,0.088081,0.090094,0.089117,0.085722
17,US,01001,NaN,NaN,NaN,2203530192,NaN,NaN,IDLING,287.365881,...,0.069847,0.083470,0.091061,0.089665,0.092922,0.089711,0.088081,0.090094,0.089117,0.085722


In [9]:
hdv_oni.head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,mar_ratio,apr_ratio,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio
6,US,01001,NaN,NaN,NaN,2201610192,NaN,NaN,IDLING,"3,507.302683",...,0.084626,0.081908,0.086663,0.084532,0.086416,0.084717,0.082921,0.087482,0.083946,0.085106
15,US,01001,NaN,NaN,NaN,2202610192,NaN,NaN,IDLING,"306,920.729700",...,0.084626,0.081908,0.086663,0.084532,0.086416,0.084717,0.082921,0.087482,0.083946,0.085106
16,US,01001,NaN,NaN,NaN,2202620192,NaN,NaN,IDLING,"68,779.579260",...,0.072164,0.080817,0.089029,0.088239,0.094130,0.092194,0.092407,0.098184,0.090324,0.081480
18,US,01001,NaN,NaN,NaN,2203610192,NaN,NaN,IDLING,15.626584,...,0.084626,0.081908,0.086663,0.084532,0.086416,0.084717,0.082921,0.087482,0.083946,0.085106
29,US,01003,NaN,NaN,NaN,2201610192,NaN,NaN,IDLING,"7,872.923717",...,0.084607,0.081931,0.086654,0.084535,0.086423,0.084705,0.082927,0.087470,0.083955,0.085103


<h1> Load LOCUS county-level idling data

In [10]:
# Load LOCUS data 
locus = gpd.read_file('LOCUS_CountyIdling.shp')
locus = locus.to_crs('EPSG:4326')

In [11]:
locus.head()

,GISJOIN,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,...,EPA_HDV_ON,EPA_HDV_HO,EPA_HDV_To,EPA_Total,DIFF_MDV,DIFF_HDV,DIFF_TOT,LOCUS_MDV_,LOCUS_HDV_,geometry
0,G1700010,17,001,00424202,17001,Adams,Adams County,06,H1,G4020,...,429.555627,276.551981,706.107608,"1,404.834634",666.134698,674.228793,"1,340.363491","498,174.528808","503,822.786543","POLYGON ((-90.91347 40.10446, -90.91349 40.101..."
1,G1700030,17,003,00424203,17003,Alexander,Alexander County,06,H1,G4020,...,116.032072,131.900436,247.932508,367.477779,-49.018173,-141.871493,-190.889666,"25,742.391106","38,712.270316","POLYGON ((-89.17208 37.06831, -89.17295 37.067..."
2,G1700050,17,005,00424204,17005,Bond,Bond County,06,H1,G4020,...,484.349349,886.603997,"1,370.953346","1,687.420908",33.867008,-370.724675,-336.857667,"127,872.118026","365,083.464960","POLYGON ((-89.25419 38.74202, -89.25419 38.741..."
3,G1700070,17,007,00424205,17007,Boone,Boone County,06,H1,G4020,...,690.685964,879.465793,"1,570.151757","2,347.147556",-230.145532,638.715161,408.569629,"199,600.347346","806,236.425044","POLYGON ((-88.70559 42.15354, -88.70568 42.153..."
4,G1700090,17,009,00424206,17009,Brown,Brown County,06,H1,G4020,...,45.016232,0.000000,45.016232,113.011141,133.821711,319.157821,452.979532,"73,663.066498","132,923.529278","POLYGON ((-90.51375 39.98789, -90.51316 39.986..."


<h1> Combine LOCUS idling with EPA data based on MOVES vehicle codes

In [12]:
## Map MDV ONI
# locus data mapped to oni file using county code (region_cd) as index 
locus["LOCUS_MDV_ann"] = locus["MDT_avg_da"] * 365  #This data is for 2024, which is a leap year but im mapping it to the 2022 inventory, so 365 is fine
locus_mdv_ann = locus.set_index("region_cd")["LOCUS_MDV_ann"] 
mdv_oni["LOCUS_MDV_ann"] = mdv_oni["region_cd"].map(locus_mdv_ann).fillna(0.0)

# logic here:
# Determine MDV ONI (52/53) total per count (den_total).
# Next, calculate of the total, per county, how much idling was distributed across vehicle code 52/53 and fuel types. 
# Then,take LOCUS county-level totals for MDV and update ann_parm_value for EPA Idling
# based on the ratios of current activity. Therefore, the general characterization is not
# modified, but total idling activity is increased proportionately. 
mdv_oni["den_total"] = mdv_oni.groupby("region_cd")["ann_parm_value"].transform("sum") #total ONI by 52/53 at county (region_cd) level
mdv_oni["weight"] = 0.0 # initialize 
nz = mdv_oni["den_total"] > 0 # Make sure is nonzero
mdv_oni.loc[nz, "weight"] = mdv_oni.loc[nz, "ann_parm_value"] / mdv_oni.loc[nz, "den_total"] #calc ratio of sinlge unit fuel type combo to total idling
if (~nz).any():
    n_rows = mdv_oni.loc[~nz].groupby("region_cd")["region_cd"].transform("count")
    mdv_oni.loc[~nz, "weight"] = 1.0 / n_rows  # equal split only when ONI has zero total

mdv_oni["LOCUS_alloc_MDV_ann"] = mdv_oni["LOCUS_MDV_ann"] * mdv_oni["weight"]  # LOCUS data allocated to EPA defs based on weights we determined above
mdv_oni["LOCUS_alloc_MDV_da"]  = mdv_oni["LOCUS_alloc_MDV_ann"] / 365.0

oni_file["LOCUS_alloc_MDV_da"] = 0.0
oni_file.loc[mdv_oni.index, "LOCUS_alloc_MDV_ann"] = mdv_oni["LOCUS_alloc_MDV_ann"].astype(float).fillna(0.0).values
oni_file.loc[mdv_oni.index, "LOCUS_alloc_MDV_da"]  = mdv_oni["LOCUS_alloc_MDV_da"].astype(float).fillna(0.0).values

In [13]:
# Map HDV 
# HDV has both ONI and hotelling idling activty to consider
# So here, I pull both the ONI and hotelling files

#Convert locus daily to annual idling
locus["LOCUS_HDV_ann"] = locus["HDT_avg_da"] * 365.0
loc_hdv_ann = locus.set_index("region_cd")["LOCUS_HDV_ann"]

# HDV here will defined as combination trucks (scc 61 or 62)
hdv_oni = oni_file[oni_file["veh_code"].isin(["61","62"])].copy()
hdv_hot = hotelling_file[hotelling_file["veh_code"].isin(["61","62"])].copy()  # typically 62 only but just to safe 

# Make EPA values numeric
for df in (hdv_oni, hdv_hot):
    df["ann_parm_value"] = pd.to_numeric(df["ann_parm_value"], errors="coerce").fillna(0.0)

# Keep a reference to original row indices for future ref
hdv_oni["row_index"] = hdv_oni.index
hdv_hot["row_index"] = hdv_hot.index

# Build a single comparison table with source flag bc HDV idling comes from both ONI and hotelling 
epa_hdv_oni = hdv_oni[["region_cd","ann_parm_value","row_index"]].copy()
epa_hdv_oni["source"] = "ONI"
epa_hdv_hot = hdv_hot[["region_cd","ann_parm_value","row_index"]].copy()
epa_hdv_hot["source"] = "HOT"
epa_hdv = pd.concat([epa_hdv_oni, epa_hdv_hot], ignore_index=True)

# logic here:  Determine MDV ONI (52/53) total per count (den_total). 
# Next, calculate of the total, per county, how much idling was distributed across vehicle code 52/53 and fuel types. 
# Then,take LOCUS county-level totals for MDV and update ann_parm_value for EPA Idling 
# based on the ratios of current activity. Therefore, the general characterization is not 
# modified, but total idling activity is increased proportionately.
epa_hdv["den_total"] = epa_hdv.groupby("region_cd")["ann_parm_value"].transform("sum")
epa_hdv["weight"] = 0.0
nz = epa_hdv["den_total"] > 0
epa_hdv.loc[nz, "weight"] = epa_hdv.loc[nz, "ann_parm_value"] / epa_hdv.loc[nz, "den_total"]
# if (~nz).any():
#     n_rows = epa_hdv.loc[~nz].groupby("region_cd")["region_cd"].transform("count")
#     epa_hdv.loc[~nz, "weight"] = 1.0 / n_rows  # equal split only when ONI has zero total

# Take the weight and scale LOCUS data accordingly 
epa_hdv["LOCUS_HDV_ann"]       = epa_hdv["region_cd"].map(loc_hdv_ann).fillna(0.0)
epa_hdv["LOCUS_alloc_HDV_ann"] = epa_hdv["LOCUS_HDV_ann"] * epa_hdv["weight"]
epa_hdv["LOCUS_alloc_HDV_da"]  = epa_hdv["LOCUS_alloc_HDV_ann"] / 365.0

# Add to original tables 
# Ensure destination columns exist
for df in (oni_file, hotelling_file):
    if "LOCUS_alloc_HDV_ann" not in df.columns: df["LOCUS_alloc_HDV_ann"] = 0.0
    if "LOCUS_alloc_HDV_da"  not in df.columns: df["LOCUS_alloc_HDV_da"]  = 0.0

sel_oni = epa_hdv[epa_hdv["source"]=="ONI"]
oni_file.loc[sel_oni["row_index"], "LOCUS_alloc_HDV_ann"] = sel_oni["LOCUS_alloc_HDV_ann"].values
oni_file.loc[sel_oni["row_index"], "LOCUS_alloc_HDV_da"]  = sel_oni["LOCUS_alloc_HDV_da"].values

sel_hot = epa_hdv[epa_hdv["source"]=="HOT"]
hotelling_file.loc[sel_hot["row_index"], "LOCUS_alloc_HDV_ann"] = sel_hot["LOCUS_alloc_HDV_ann"].values
hotelling_file.loc[sel_hot["row_index"], "LOCUS_alloc_HDV_da"]  = sel_hot["LOCUS_alloc_HDV_da"].values

In [14]:
print(locus.loc[locus['region_cd']=='17031']['HDT_avg_da']*365)
print(locus.loc[locus['region_cd']=='17031']['MDT_avg_da']*365)

15   42,070,495.203683
Name: HDT_avg_da, dtype: float64
15   27,197,197.281205
Name: MDT_avg_da, dtype: float64


In [15]:
# Including print statements for sanity checks and hand calculations to check work
oni_file.loc[(oni_file['veh_code']=='52') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_MDV_da,LOCUS_alloc_MDV_ann,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
11858,US,17031,NaN,NaN,NaN,2201520192,NaN,NaN,IDLING,"5,263,398.530000",...,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220,"26,149.302421","9,544,495.383596",0.000000,0.000000
11868,US,17031,NaN,NaN,NaN,2202520192,NaN,NaN,IDLING,"8,632,081.771000",...,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220,"42,885.393432","15,653,168.602860",0.000000,0.000000
11874,US,17031,NaN,NaN,NaN,2203520192,NaN,NaN,IDLING,"133,976.576900",...,0.086954,0.088734,0.083434,0.088967,0.079182,0.077220,665.614433,"242,949.267939",0.000000,0.000000


In [17]:
oni_file.loc[(oni_file['veh_code']=='53') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_MDV_da,LOCUS_alloc_MDV_ann,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
11859,US,17031,NaN,NaN,NaN,2201530192,NaN,NaN,IDLING,"275,409.436300",...,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217,"1,368.272723","499,419.543928",0.000000,0.000000
11869,US,17031,NaN,NaN,NaN,2202530192,NaN,NaN,IDLING,"687,557.819200",...,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217,"3,415.883719","1,246,797.557492",0.000000,0.000000
11875,US,17031,NaN,NaN,NaN,2203530192,NaN,NaN,IDLING,"5,716.935015",...,0.086953,0.088733,0.083431,0.088970,0.079185,0.077217,28.402535,"10,366.925390",0.000000,0.000000


In [18]:
oni_file.loc[(oni_file['veh_code']=='61') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_MDV_da,LOCUS_alloc_MDV_ann,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
11860,US,17031,NaN,NaN,NaN,2201610192,NaN,NaN,IDLING,"4,243.243977",...,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085,0.000000,NaN,"14,730.838028",40.358460
11870,US,17031,NaN,NaN,NaN,2202610192,NaN,NaN,IDLING,"4,871,835.012000",...,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085,0.000000,NaN,"16,913,053.514948","46,337.132918"
11876,US,17031,NaN,NaN,NaN,2203610192,NaN,NaN,IDLING,"49,991.035370",...,0.084635,0.088145,0.083872,0.087436,0.079377,0.075085,0.000000,NaN,"173,548.786935",475.476129


In [19]:
oni_file.loc[(oni_file['veh_code']=='62') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_MDV_da,LOCUS_alloc_MDV_ann,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
11871,US,17031,NaN,NaN,NaN,2202620192,NaN,NaN,IDLING,"3,311,731.344000",...,0.083827,0.088684,0.082185,0.090296,0.083080,0.073850,0.000000,NaN,"11,497,000.475229","31,498.631439"


In [20]:
oni_file.loc[(oni_file['veh_code']=='62') & (oni_file['region_cd']=='17031')].head()

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_MDV_da,LOCUS_alloc_MDV_ann,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
11871,US,17031,NaN,NaN,NaN,2202620192,NaN,NaN,IDLING,"3,311,731.344000",...,0.083827,0.088684,0.082185,0.090296,0.083080,0.073850,0.000000,NaN,"11,497,000.475229","31,498.631439"


<h1> Update annual idling values and monthly idling activitiy 

In [21]:
# Last redundancy I hope 
oni_file["LOCUS_alloc_MDV_ann"] = pd.to_numeric(oni_file["LOCUS_alloc_MDV_da"], errors="coerce").fillna(0.0) * 365.0
oni_file["LOCUS_alloc_HDV_ann"] = pd.to_numeric(oni_file["LOCUS_alloc_HDV_da"], errors="coerce").fillna(0.0) * 365.0
hotelling_file["LOCUS_alloc_HDV_ann"] = pd.to_numeric(hotelling_file["LOCUS_alloc_HDV_da"], errors="coerce").fillna(0.0) * 365.0

# checks for numeric
for df, col in (
    (oni_file,"ann_parm_value"),
    (hotelling_file,"ann_parm_value"),
    (oni_file,"LOCUS_alloc_MDV_ann"),
    (oni_file,"LOCUS_alloc_HDV_ann"),
    (hotelling_file,"LOCUS_alloc_HDV_ann"),
):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# Illinois masks
mask_IL_oni = oni_file["region_cd"].str.startswith("17")
mask_IL_hot = hotelling_file["region_cd"].str.startswith("17")

# I filled any values we did not edit with 0s, so here we can use this as a condition 
# any row where idling is >0 , lets overwrite the activity data with the scaled
# locus idling estimates for ONI and Hotelling 
mdv_mask = mask_IL_oni & (oni_file.get("LOCUS_alloc_MDV_ann", 0) > 0)
hdv_mask = mask_IL_oni & (oni_file.get("LOCUS_alloc_HDV_ann", 0) > 0)

oni_file.loc[mdv_mask, "ann_parm_value"] = oni_file.loc[mdv_mask, "LOCUS_alloc_MDV_ann"].values
oni_file.loc[hdv_mask, "ann_parm_value"] = oni_file.loc[hdv_mask, "LOCUS_alloc_HDV_ann"].values

hot_mask = mask_IL_hot & (hotelling_file.get("LOCUS_alloc_HDV_ann", 0) > 0)
hotelling_file.loc[hot_mask, "ann_parm_value"] = hotelling_file.loc[hot_mask, "LOCUS_alloc_HDV_ann"].values

# ONI
for m in month_cols:
    r = m.replace("_value", "_ratio")
    ratios = pd.to_numeric(oni_file[r], errors="coerce").fillna(0.0)
    oni_file[m] = ratios * oni_file["ann_parm_value"]

resid = oni_file["ann_parm_value"] - oni_file[month_cols].sum(axis=1)
oni_file["dec_value"] = oni_file["dec_value"] + resid.fillna(0.0)

# HOTELLING (same logic)
for m in month_cols:
    r = m.replace("_value", "_ratio")
    ratios = pd.to_numeric(hotelling_file[r], errors="coerce").fillna(0.0)
    hotelling_file[m] = ratios * hotelling_file["ann_parm_value"]

resid_hot = hotelling_file["ann_parm_value"] - hotelling_file[month_cols].sum(axis=1)
hotelling_file["dec_value"] = hotelling_file["dec_value"] + resid_hot.fillna(0.0)

In [18]:
hotelling_file.loc[hotelling_file['region_cd']=='17031']

,country_cd,region_cd,tribal_code,census_tract_cd,shape_id,scc,act_parm_type_cd,act_parm_uofmsr,activity_type,ann_parm_value,...,may_ratio,jun_ratio,jul_ratio,aug_ratio,sep_ratio,oct_ratio,nov_ratio,dec_ratio,LOCUS_alloc_HDV_ann,LOCUS_alloc_HDV_da
469,US,17031,NaN,NaN,NaN,2202620153,NaN,NaN,HOTELLING,"3,500,373.360000",...,0.086941,0.087149,0.083849,0.088669,0.082181,0.090221,0.082809,0.073965,"12,151,889.752866","33,292.848638"
470,US,17031,NaN,NaN,NaN,2202620191,NaN,NaN,HOTELLING,"380,306.640000",...,0.086941,0.087149,0.083849,0.088669,0.082181,0.090221,0.082809,0.073965,"1,320,271.835677","3,617.183111"


<h1> Export data to CSV file, maintaining original file structure 

In [19]:
# EPA files have quotes around the several columns -- so ensuring this format is used
# Also, the deciles are 
def escape_csv_field(val: str) -> str:
    """Minimal CSV escaping for non-forced-quote fields."""
    s = "" if pd.isna(val) else str(val)
    if any(ch in s for ch in [',', '"', '\n', '\r']):
        s = '"' + s.replace('"', '""') + '"'
    return s

def force_quote(s: str) -> str:
    """Return value wrapped in double quotes, doubling internal quotes."""
    s = "" if pd.isna(s) else str(s)
    return '"' + s.replace('"', '""') + '"'

def write_with_metadata_body(
    src_path: str,
    out_path: str,
    meta_lines: int,
    df_body: pd.DataFrame,
    line_ending: str = "\n",     
):

    with open(src_path, "r", encoding="utf-8", newline="") as f:
        meta = [next(f) for _ in range(meta_lines)]

    body = df_body[cols].copy()

    # make sure each col is always 5 and 10 digits long - fill with leading 0s if not. 
    body["region_cd"] = body["region_cd"].astype(str).str.zfill(5)
    body["scc"]       = body["scc"].astype(str).str.zfill(10)

    # Convert all to strings (so we control formatting)
    for c in cols:
        if c not in body.columns:
            body[c] = ""
        body[c] = body[c].astype(object).where(pd.notna(body[c]), "")

    # Columns that must ALWAYS be quoted per EPA formatting
    always_quote = {"country_cd", "region_cd", "scc","activity_type","date_updated"}

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        # metadata first, copy/paste from orig files
        for line in meta:
            if line_ending != "\n":
                line = line.rstrip("\r\n") + line_ending
            f.write(line)

        for _, row in body.iterrows():
            fields = []
            for c in cols:
                if c == "country_cd":
                    # force literal "US" 
                    fields.append('"US"')
                elif c in always_quote:
                    fields.append(force_quote(row[c]))
                else:
                    fields.append(escape_csv_field(row[c]))
            f.write(",".join(fields) + line_ending)

oni_src_path   = "ONI_2022v1_projected_from_full_annual_20240226_monthly_18jun2024_nf_v5.csv"
hot_src_path   = "HOTELING_2022v1_monthly_20240305_18jun2024_nf_v5.csv"
oni_out        = "ONI_2022v1_LOCUS_scaled.csv"
hot_out        = "HOTELING_2022v1_LOCUS_scaled.csv"
oni_meta_lines = 14
hot_meta_lines = 19  # set to the true preamble line count for your HOT file

# write_with_metadata_body(oni_src_path, oni_out, oni_meta_lines, oni_file)
#write_with_metadata_body(hot_src_path, hot_out, hot_meta_lines, hotelling_file)

print("Wrote:")
print(" ", oni_out)
print(" ", hot_out)

Wrote:
  ONI_2022v1_LOCUS_scaled.csv
  HOTELING_2022v1_LOCUS_scaled.csv


<h1> Calculate % of Idling attributable to APUs

In [21]:
# --- County-level share of hoteling done by APU vs extended idle for CMAP counties ---

# CMAP county FIPS codes
cmap_fips = ["17031", "17043", "17089", "17093", "17097", "17111", "17197"]

# make sure key fields are strings / numeric
hotelling_file["region_cd"] = hotelling_file["region_cd"].astype(str)
hotelling_file["ann_parm_value"] = pd.to_numeric(
    hotelling_file["ann_parm_value"], errors="coerce"
).fillna(0.0)

# assumes process ID is stored in the last two digits of SCC
scc_hot = hotelling_file["scc"].astype(str).str.zfill(10)
hotelling_file["process_id"] = scc_hot.str[-2:]

# filter to CMAP counties and the two hoteling process IDs
county_idle_pct = (
    hotelling_file[
        hotelling_file["region_cd"].isin(cmap_fips)
        & hotelling_file["process_id"].isin(["53", "91"])
    ]
    .groupby(["region_cd", "process_id"])["ann_parm_value"]
    .sum()
    .unstack(fill_value=0)
    .rename(columns={"91": "apu_idle_ann", "53": "extended_idle_ann"})
)

# total and percentages
county_idle_pct["total_idle_ann"] = (
    county_idle_pct.get("apu_idle_ann", 0) +
    county_idle_pct.get("extended_idle_ann", 0)
)

county_idle_pct["pct_apu_idle"] = np.where(
    county_idle_pct["total_idle_ann"] > 0,
    100 * county_idle_pct.get("apu_idle_ann", 0) / county_idle_pct["total_idle_ann"],
    0
)

county_idle_pct["pct_extended_idle"] = np.where(
    county_idle_pct["total_idle_ann"] > 0,
    100 * county_idle_pct.get("extended_idle_ann", 0) / county_idle_pct["total_idle_ann"],
    0
)

county_idle_pct = county_idle_pct.reset_index()

county_idle_pct

process_id,region_cd,extended_idle_ann,apu_idle_ann,total_idle_ann,pct_apu_idle,pct_extended_idle
0,17031,"3,500,373.360000","380,306.640000","3,880,680.000000",9.800000,90.200000
1,17043,"1,741,196.002200","189,176.505780","1,930,372.507980",9.800000,90.200000
2,17089,"845,533.610530","91,865.070767","937,398.681297",9.800000,90.200000
3,17097,"793,175.536110","86,176.499489","879,352.035599",9.800000,90.200000
4,17111,"94,818.240000","10,301.760000","105,120.000000",9.800000,90.200000
5,17197,"1,343,051.949300","145,919.169660","1,488,971.118960",9.800000,90.200000
